#### Create Catalog, Volumes and Schemas

In [0]:
spark.sql("create catalog if not exists airline_analytics")

In [0]:
spark.sql("use catalog airline_analytics")

In [0]:
spark.sql("create schema if not exists bronze")
spark.sql("create schema if not exists silver")
spark.sql("create schema if not exists gold")

In [0]:
spark.sql(""" create volume if not exists airline_analytics.bronze.landing
          
comment 'Raw files landed manually (recent BTS montly extracts) before ingestion into Bronze'
          """)

print("Catalog, schemas and landing volume ready.")

In [0]:
display(spark.sql("show schemas in airline_analytics"))

### Read the built-in data and write to Bronze

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import lit, current_timestamp, input_file_name, col

In [0]:

airline_schema = StructType([
    StructField("Year", IntegerType()),
    StructField("Month", IntegerType()),
    StructField("DayofMonth", IntegerType()),
    StructField("DayOfWeek", IntegerType()),
    StructField("DepTime", IntegerType()),
    StructField("CRSDepTime", IntegerType()),
    StructField("ArrTime", IntegerType()),
    StructField("CRSArrTime", IntegerType()),
    StructField("UniqueCarrier", StringType()),
    StructField("FlightNum", StringType()),
    StructField("TailNum", StringType()),
    StructField("ActualElapsedTime", IntegerType()),
    StructField("CRSElapsedTime", IntegerType()),
    StructField("AirTime", IntegerType()),
    StructField("ArrDelay", IntegerType()),
    StructField("DepDelay", IntegerType()),
    StructField("Origin", StringType()),
    StructField("Dest", StringType()),
    StructField("Distance", IntegerType()),
    StructField("TaxiIn", IntegerType()),
    StructField("TaxiOut", IntegerType()),
    StructField("Cancelled", IntegerType()),
    StructField("CancellationCode", StringType()),
    StructField("Diverted", IntegerType()),
    StructField("CarrierDelay", IntegerType()),
    StructField("WeatherDelay", IntegerType()),
    StructField("NASDelay", IntegerType()),
    StructField("SecurityDelay", IntegerType()),
    StructField("LateAircraftDelay", IntegerType()),
    StructField("IsArrDelayed", StringType()),
    StructField("IsDepDelayed", StringType()),
])

historical_raw = (
    spark.read
    .option("header", "false")     # <-- no real header in these files
    .schema(airline_schema)        # <-- explicit schema instead of inference
    .option("nullValue", "NA")
    .csv("dbfs:/databricks-datasets/airlines/part-*")
)

print(historical_raw.columns)

In [0]:
#Filter the desired years
YEARS_TO_LOAD = [2004, 2005, 2006, 2007, 2008]

historical_filtered = historical_raw.filter(historical_raw.Year.isin(YEARS_TO_LOAD))

In [0]:
from pyspark.sql.functions import lit, current_timestamp, col

historical_deduped = historical_filtered.dropDuplicates([
    "Year", "Month", "DayofMonth", "UniqueCarrier",
    "FlightNum", "DepTime", "Origin", "Dest"
])

historical_bronze = (
    historical_deduped
    .withColumn("_source_system", lit("databricks_datasets_airlines"))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
#write to bronze
(historical_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("airline_analytics.bronze.flights_historical")
)

print(f"bronze.flights_historical loaded: {spark.table('airline_analytics.bronze.flights_historical').count():,} rows")

### Debugging

In [0]:
print(dbutils.fs.head("dbfs:/databricks-datasets/airlines/README.md", 2000))

In [0]:
%sql
-- distinct files actually read vs files that exist
SELECT COUNT(DISTINCT _source_file) AS distinct_files
FROM airline_analytics.bronze.flights_historical

In [0]:
%sql
-- per-year counts
SELECT Year, COUNT(*) AS row_count
FROM airline_analytics.bronze.flights_historical
GROUP BY Year ORDER BY Year

In [0]:
%sql
-- duplicate rows check
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT Year, Month, DayofMonth, UniqueCarrier, FlightNum, DepTime, Origin, Dest) AS distinct_flights
FROM airline_analytics.bronze.flights_historical

In [0]:
%sql
SELECT _source_file, COUNT(*) AS occurrences
FROM airline_analytics.bronze.flights_historical
WHERE Year = 2004 AND Month = 1 AND DayofMonth = 1
  AND UniqueCarrier = 'AA'
GROUP BY _source_file
ORDER BY occurrences DESC

In [0]:
raw_count = historical_filtered.count()
deduped_count = historical_deduped.count()
print(f"Raw rows: {raw_count:,}")
print(f"After dedup: {deduped_count:,}")
print(f"Duplicates removed: {raw_count - deduped_count:,} ({(1 - deduped_count/raw_count)*100:.1f}%)")


In [0]:
%sql
SELECT COUNT(*) FROM airline_analytics.bronze.flights_historical WHERE DepTime IS NULL

### Read Recent data from volume (Uploaded data)

In [0]:
from pyspark.sql.functions import lit, current_timestamp, col

recent_raw = (spark.read
.option("header", "true")
    .option("inferSchema", "true")
    .option("mergeSchema", "true")   # BTS schema has drifted slightly across 2009-2026
    .csv("/Volumes/airline_analytics/bronze/landing/")
)

print(f"Columns: {len(recent_raw.columns)}")   # should be 109 (or 110 if trailing-comma quirk is present)
print(recent_raw.columns)

In [0]:
recent_bronze = (
    recent_raw
    .withColumn("_source_system", lit("bts_transtats_reporting_carrier"))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

(
    recent_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .saveAsTable("airline_analytics.bronze.flights_recent")
)

print(f"bronze.flights_recent loaded: {spark.table('airline_analytics.bronze.flights_recent').count():,} rows")

#### Data Quality validation

In [0]:
%sql
-- per-year counts: should show a rough upward trend, no wild outliers
SELECT Year, COUNT(*) AS row_count
FROM airline_analytics.bronze.flights_recent
GROUP BY Year ORDER BY Year

In [0]:
# confirm no duplicate file reads snuck in from the multiple upload retries
print(f"Distinct source files: ", end="")
display(spark.sql("SELECT COUNT(DISTINCT _source_file) AS n FROM airline_analytics.bronze.flights_recent"))
# should be 210, matching your uploaded file count

print(f"Column count: {len(spark.table('airline_analytics.bronze.flights_recent').columns)}")
# should be 109 + 3 metadata columns = 112

In [0]:
ingested_files = set(row['_source_file'] for row in 
    spark.sql("SELECT DISTINCT _source_file FROM airline_analytics.bronze.flights_recent").collect())

all_files = set(f.path for f in dbutils.fs.ls("dbfs:/Volumes/airline_analytics/bronze/landing/"))

missing = all_files - ingested_files
print(missing)